# Workshop 2: Scikit-learn and the ML Workflow

## 1. Overview & Setup

### 1.1 Overview
This workshop introduces **Scikit-learn**, the reference Python library for classical machine learning. We will execute the eight steps of an ML workflow on a classification example (Wisconsin Breast Cancer dataset) and a regression example (synthetic server response time).

### 1.2 Setup & Imports
We import the necessary libraries: `numpy`, `pandas`, `matplotlib`, and `sklearn`.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

print("Scikit-learn version:", sklearn.__version__)

Scikit-learn version: 1.6.1


## 2. Part 1: Scikit-learn

### 2.1 The Estimator Interface
Scikit-learn provides a uniform estimator interface with core methods:
- `model = Algorithm(hyper_parameters)`: Select model family & hyper-parameters.
- `model.fit(X_train, y_train)`: Learn parameters from training data.
- `model.predict(X_new)`: Predict class or value for new data.
- `model.predict_proba(X_new)`: Output class probabilities (classification).
- `model.score(X_test, y_test)`: Compute default performance metric.
- `scaler.fit_transform(X_train)` / `scaler.transform(X_test)`: Preprocessing.

### 2.2 The 8 Steps of the ML Workflow Mapped to Tools
1. **Define the problem**: Identify input features $X$, target $y$, and problem type.
2. **Collect and explore**: `pd.DataFrame`, `describe()`, `value_counts()`, plots.
3. **Prepare**: `train_test_split`, `StandardScaler`, `OneHotEncoder`.
4. **Select a model**: `LogisticRegression()`, `DecisionTreeClassifier()`, etc.
5. **Train**: `model.fit(X_train, y_train)`.
6. **Evaluate**: `confusion_matrix`, `precision_score`, `recall_score`, `r2_score`.
7. **Tune**: `cross_val_score`, `GridSearchCV`, `predict_proba`.
8. **Deploy & Monitor**: `joblib.dump` / `joblib.load`, prediction wrapper function.


## 3. Part 2: The Eight Steps on a Classification Example

### Step 1: Define the Problem
- **Input $X$**: 30 tumour measurements (radius, texture, area, etc.).
- **Output $y$**: Binary classification — Benign (0) vs. Malignant (1, positive class).
- **Goal**: Catch malignant tumours. Missing a malignant tumour (False Negative) is worse than a false alarm (False Positive).


### Step 2: Load and Explore the Data


In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(1 - data.target, name="malignant")

print("Dataset shape:", X.shape)
print("Class distribution:")
print(y.value_counts())
print("First 5 feature statistics:")
print(X.iloc[:, :5].describe().round(1))
print("Mean values by class:")
print(X.groupby(y).mean().iloc[:, :5].round(1))

### Step 3: Prepare (Split then Scale)

**The Golden Rule**: Always split the dataset into training and test sets *before* applying scaling. Scaling parameters (mean and standard deviation) must be calculated exclusively on the training set to prevent **data leakage**.


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

NameError: name 'X' is not defined

### Steps 4 & 5: Choose a Model and Train It


In [4]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)
print("Train accuracy:", round(model.score(X_train_s, y_train), 3))
print("Test accuracy:", round(model.score(X_test_s, y_test), 3))

NameError: name 'X_train_s' is not defined

### Step 6: Evaluate with Metrics


In [5]:
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             precision_score, recall_score, f1_score)

y_pred = model.predict(X_test_s)
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
print("Confusion matrix:\n", cm)
ConfusionMatrixDisplay(cm, display_labels=["benign", "malignant"]).plot()
plt.title("Logistic Regression - Confusion Matrix")
plt.show()
print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall:", round(recall_score(y_test, y_pred), 3))
print("F1 score:", round(f1_score(y_test, y_pred), 3))

NameError: name 'X_test_s' is not defined

### Step 7: Tune Threshold and Compare Algorithms

#### 7.1 Moving Decision Threshold


In [6]:
y_prob = model.predict_proba(X_test_s)[:, 1]
for threshold in [0.3, 0.5, 0.7]:
    yp = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, yp, labels=[0, 1]).ravel()
    print(f"Threshold {threshold:.1f}: FN={fn}, FP={fp}, cost={20*fn + fp}")

NameError: name 'X_test_s' is not defined

#### 7.2 Compare Algorithms via Cross-Validation Pipeline


In [7]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

candidates = {
    "logistic regression": LogisticRegression(max_iter=1000),
    "decision tree": DecisionTreeClassifier(random_state=42),
    "k-nearest neighbours": KNeighborsClassifier(5),
    "random forest": RandomForestClassifier(random_state=42)
}

for name, algo in candidates.items():
    pipe = make_pipeline(StandardScaler(), algo)
    scores = cross_val_score(pipe, X_train, y_train, cv=5)
    print(f"{name:22s} mean accuracy: {scores.mean():.3f}")


NameError: name 'X_train' is not defined

### Step 8: Save, Reload, and Inference


In [8]:
import joblib

final = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
final.fit(X_train, y_train)
print("Final test accuracy:", round(final.score(X_test, y_test), 3))
joblib.dump(final, "tumor_model.joblib")
loaded_model = joblib.load("tumor_model.joblib")
new_case = X_test.iloc[[0]]
print("Predicted class:", loaded_model.predict(new_case)[0])
print("Probabilities [benign, malignant]:", loaded_model.predict_proba(new_case)[0].round(3))
print("True class:", int(y_test.iloc[0]))

NameError: name 'X_train' is not defined

## 4. Part 3: The Same Workflow for Regression

We apply the identical workflow steps to a regression problem predicting server response time based on system load.


In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

rng = np.random.default_rng(0)
n = 300
users = rng.integers(50, 1200, n)
cpu = np.clip(0.06 * users + rng.normal(0, 8, n), 2, 100).round(1)
resp = (40 + 0.25 * users + 1.5 * cpu + rng.normal(0, 25, n)).round(8)
srv = pd.DataFrame({"users": users, "cpu": cpu, "response_ms": resp})

Xr, yr = srv[["users", "cpu"]], srv["response_ms"]
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    Xr, yr, test_size=0.2, random_state=42
)
reg = LinearRegression().fit(Xr_train, yr_train)
pred = reg.predict(Xr_test)
rmse = np.sqrt(mean_squared_error(yr_test, pred))
r2 = r2_score(yr_test, pred)
print("RMSE:", round(rmse, 1), "ms")
print("R2:", round(r2, 3))
print(f"Equation: response = {reg.intercept_:.1f} + {reg.coef_[0]:.3f} * users + {reg.coef_[1]:.3f} * cpu")
sample_pred = reg.predict(pd.DataFrame([[800, 60]], columns=["users", "cpu"]))
print(f"Prediction: {sample_pred[0]:.0f} ms for 800 users at 60% CPU")

RMSE: 21.3 ms
R2: 0.968
Equation: response = 37.1 + 0.242 * users + 1.668 * cpu
Prediction: 331 ms for 800 users at 60% CPU


## 5. Part 4: Exercises, Checkpoint Questions & Homework

### 5.1 Exercises

#### Exercise 1: Model Comparison with Naive Bayes & SVC
In Cell 2.6, add `GaussianNB()` (from `sklearn.naive_bayes import GaussianNB`) and `SVC()` (from `sklearn.svm import SVC`) to the `candidates` dictionary and evaluate cross-validation accuracy. Populate `ex1_scores` dictionary with mean CV scores.


In [10]:
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

candidates["gaussian naive bayes"] = GaussianNB()
candidates["svc"] = SVC()

ex1_scores = {}
for name, algorithm in candidates.items():
    pipeline = make_pipeline(StandardScaler(), algorithm)
    scores = cross_val_score(pipeline, X_train, y_train, cv=5)
    ex1_scores[name] = scores.mean()

for name, score in ex1_scores.items():
    print(f"{name:22s}: {score:.3f}")

NameError: name 'X_train' is not defined

In [ ]:
# nbgrader test: Exercise 1 verification Skip it !

In [ ]:
# Write your code here

#### Exercise 2: Logistic Regression Without Scaling
Retrain Logistic Regression without scaling (`LogisticRegression(max_iter=1000).fit(X_train, y_train)`). Compute test accuracy as `unscaled_test_acc`.


In [11]:
import warnings
from sklearn.exceptions import ConvergenceWarning

unscaled_model = LogisticRegression(max_iter=1000)
# Convergence may be slower without scaling; warning is informative.
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", ConvergenceWarning)
    unscaled_model.fit(X_train, y_train)

unscaled_test_acc = unscaled_model.score(X_test, y_test)
scaled_test_acc = model.score(X_test_s, y_test)
print("Without scaling:", round(unscaled_test_acc, 3))
print("With scaling:   ", round(scaled_test_acc, 3))
if any(issubclass(w.category, ConvergenceWarning) for w in caught):
    print("Note: the unscaled model raised a convergence warning.")

NameError: name 'X_train' is not defined

In [ ]:
# nbgrader test: Exercise 2 verification skip it !

### Exercise 3
A hospital says a missed malignant tumour costs 20 times more than a false alarm. Which threshold of Cell 2.5 would you choose, and why?

In [12]:
threshold_costs = {}
for threshold in [0.3, 0.5, 0.7]:
    yp = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, yp, labels=[0, 1]).ravel()
    threshold_costs[threshold] = 20 * fn + fp
    print(f"t={threshold}: FN={fn}, FP={fp}, total cost={threshold_costs[threshold]}")

t = min(threshold_costs, key=threshold_costs.get)
because = ("I choose the threshold with the lowest cost, "
           "where a missed malignant case costs 20 times a false alarm.")
print("Chosen threshold:", t)
print("Reason:", because)

NameError: name 'y_prob' is not defined

In [ ]:
# Cell to evaluate the threshold ... skip it !

Q1. In which order: split or scale? Why?

In [13]:
answer_q1 = "Split first, then fit the scaler only on training data to avoid data leakage."
print("Q1:", answer_q1)

Q1: Split first, then fit the scaler only on training data to avoid data leakage.


Q2. Your model has accuracy 0.99 on a dataset where 1% of the rows are attacks.
Is it a good model?
Which metric do you look at next?

In [ ]:
good_model = False
next_metrics = "Check recall for the attack class, precision, F1 score and the confusion matrix."
print("Q2: Good model?", good_model)
print("Q2: Next metrics:", next_metrics)

 **Q3. What is the difference between `predict` and `predict_proba`?**

In [14]:
response = ("predict returns a class label (0 or 1); "
            "predict_proba returns probabilities for each class.")
print("Q3:", response)

Q3: predict returns a class label (0 or 1); predict_proba returns probabilities for each class.


- **Q4. Give the regression equivalents of `LogisticRegression`, `accuracy` and `confusion_matrix`.**

In [15]:
answer_q4 = ("LinearRegression for a numeric target; RMSE or R2 to evaluate performance; "
             "residual plots and error metrics instead of a confusion matrix.")
print("Q4:", answer_q4)

Q4: LinearRegression for a numeric target; RMSE or R2 to evaluate performance; residual plots and error metrics instead of a confusion matrix.


### 5.2 Homework: ML Workflow on Login Logs

Apply the eight steps to logins log data:
- Define target: `suspicious = failed_attempts >= 5`.
- Features: `hour`, `duration_min`, `night`, and one-hot encoded `country`.
- Train Logistic Regression and Decision Tree.
- Compute confusion matrix and recall on the test set.


In [2]:
# Step 1 & 2: Generate the data and define the problem
np.random.seed(42)
n_samples = 500
logins_clean = pd.DataFrame({
    "failed_attempts": np.random.randint(0, 10, size=n_samples),
    "hour": np.random.randint(0, 24, size=n_samples),
    "duration_min": np.random.exponential(scale=15, size=n_samples).round(1),
    "night": np.random.choice([0, 1], size=n_samples),
    "country": np.random.choice(["US", "FR", "CN", "RU", "DE"], size=n_samples),
})
logins_clean["suspicious"] = (logins_clean["failed_attempts"] >= 5).astype(int)
print(logins_clean.head())
print("Class distribution:")
print(logins_clean["suspicious"].value_counts())

# Step 3: Select features and convert country to dummy variables
X_hw = logins_clean[["hour", "duration_min", "night", "country"]]
X_hw = pd.get_dummies(X_hw, columns=["country"], dtype=int)
y_hw = logins_clean["suspicious"]
X_hw_train, X_hw_test, y_hw_train, y_hw_test = train_test_split(
    X_hw, y_hw, test_size=0.2, random_state=42, stratify=y_hw
)
scaler_hw = StandardScaler()
X_hw_train_s = scaler_hw.fit_transform(X_hw_train)
X_hw_test_s = scaler_hw.transform(X_hw_test)

# Steps 4 & 5: Train two models
log_reg_hw = LogisticRegression(max_iter=1000)
log_reg_hw.fit(X_hw_train_s, y_hw_train)
tree_hw = DecisionTreeClassifier(random_state=42)
tree_hw.fit(X_hw_train, y_hw_train)

# Step 6: Evaluate both models on the same test split
y_pred_log_hw = log_reg_hw.predict(X_hw_test_s)
y_pred_tree_hw = tree_hw.predict(X_hw_test)
cm_log_hw = confusion_matrix(y_hw_test, y_pred_log_hw, labels=[0, 1])
cm_tree_hw = confusion_matrix(y_hw_test, y_pred_tree_hw, labels=[0, 1])
recall_log_hw = recall_score(y_hw_test, y_pred_log_hw, zero_division=0)
recall_tree_hw = recall_score(y_hw_test, y_pred_tree_hw, zero_division=0)

print("Logistic Regression - Recall:", round(recall_log_hw, 3))
print("Decision Tree - Recall:      ", round(recall_tree_hw, 3))
print("Logistic Regression confusion matrix:\n", cm_log_hw)
print("Decision Tree confusion matrix:\n", cm_tree_hw)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay(cm_log_hw, display_labels=["normal", "suspicious"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Logistic Regression")
ConfusionMatrixDisplay(cm_tree_hw, display_labels=["normal", "suspicious"]).plot(ax=axes[1], colorbar=False)
axes[1].set_title("Decision Tree")
plt.tight_layout()
plt.show()

   failed_attempts  hour  duration_min  night country  suspicious
0                6     8          25.4      1      DE           1
1                3    16           8.7      0      CN           0
2                7    16           7.1      1      RU           1
3                4    19           9.3      1      DE           0
4                6    15           5.4      1      US           1
Class distribution:
suspicious
0    257
1    243
Name: count, dtype: int64


NameError: name 'train_test_split' is not defined

In [ ]:
# nbgrader test: Homework verification skip it !

In [16]:
from sklearn.model_selection import GridSearchCV

log_pipeline_hw = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
search_hw = GridSearchCV(log_pipeline_hw, {"logisticregression__C": [0.1, 1, 10]},
                         cv=5, scoring="recall")
search_hw.fit(X_hw_train, y_hw_train)
print("Best C:", search_hw.best_params_)
print("Cross-validation recall:", round(search_hw.best_score_, 3))
print("Tuned model test recall:", round(recall_score(y_hw_test, search_hw.predict(X_hw_test), zero_division=0), 3))

NameError: name 'X_hw_train' is not defined

In [17]:
joblib.dump(search_hw.best_estimator_, "login_model.joblib")
loaded_login_model = joblib.load("login_model.joblib")
new_login = X_hw_test.iloc[[0]]
print("Prediction:", loaded_login_model.predict(new_login)[0])
print("True label:", int(y_hw_test.iloc[0]))

AttributeError: 'GridSearchCV' object has no attribute 'best_estimator_'

**Homework Limitations Discussion:**
Expect a weak model because the underlying data is generated randomly without true statistical dependency between inputs and the target. This highlights that machine learning models cannot extract non-existent signal from uninformative noise, confirming that strictly following the 8-step ML procedure allows us to reliably assess model viability.
